In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# Settings
rng = np.random.default_rng(20)

START_DATE = "2024-05-01"
END_DATE = "2024-05-31"

NUMBER_OF_PATIENTS = 500

OUTPUT_FILE = Path(
    r"D:\ERA 5 Data\ERA5_Camden\synthetic_patient_visits.csv"
)

# ZIP codes are illustrative assignments, not population estimates.
zipcodes = ["08102", "08103", "08104", "08105"]

# Simplified synthetic demographic categories
age_groups = ["0–17", "18–44", "45–64", "65+"]
sex_categories = ["Female", "Male", "Unknown"]
race_categories = ["White", "Black", "Asian", "Other", "Unknown"]

# These are invented example notes, not clinical assessments.
notes = [
    "Synthetic visit: headache",
    "Synthetic visit: dizziness",
    "Synthetic visit: fatigue",
    "Synthetic visit: nausea",
    "Synthetic visit: respiratory symptoms",
    "Synthetic visit: minor injury",
    "Synthetic visit: other complaint",
]

# 1. Create a pool of fictional patients
patients = pd.DataFrame({
    "patient_id": [
        f"SYN{i:06d}"
        for i in range(1, NUMBER_OF_PATIENTS + 1)
    ],
    "zipcode": rng.choice(zipcodes, NUMBER_OF_PATIENTS),
    "age_group": rng.choice(age_groups, NUMBER_OF_PATIENTS),
    "sex": rng.choice(sex_categories, NUMBER_OF_PATIENTS),
    "race": rng.choice(race_categories, NUMBER_OF_PATIENTS),
})

patients["demographic_information"] = (
    "Age group: " + patients["age_group"]
    + "; sex: " + patients["sex"]
    + "; race: " + patients["race"]
)

# 2. Generate daily visits
records = []

for date in pd.date_range(START_DATE, END_DATE, freq="D"):

    # Invent 5–15 visits for each day
    number_of_visits = int(rng.integers(5, 16))

    # A patient can return on another date.
    # Each patient appears at most once on a given date.
    patient_indices = rng.choice(
        NUMBER_OF_PATIENTS,
        size=number_of_visits,
        replace=False
    )

    for index in patient_indices:
        patient = patients.iloc[index]

        records.append({
            "date": date.strftime("%Y-%m-%d"),
            "patient_id": patient["patient_id"],
            "zipcode": patient["zipcode"],
            "diagnosis_notes": rng.choice(notes),
            "age_group": patient["age_group"],
            "sex": patient["sex"],
            "race": patient["race"],
        })

# 3. Create and save the table
health_data = pd.DataFrame(records)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
health_data.to_csv(OUTPUT_FILE, index=False)

print("SYNTHETIC DATA ONLY")
print("Saved to:", OUTPUT_FILE)
print("Number of visits:", len(health_data))
print(health_data.head())

SYNTHETIC DATA ONLY
Saved to: D:\ERA 5 Data\ERA5_Camden\synthetic_patient_visits.csv
Number of visits: 294
         date patient_id zipcode            diagnosis_notes age_group     sex  \
0  2024-05-01  SYN000399   08102  Synthetic visit: headache     45–64  Female   
1  2024-05-01  SYN000102   08103   Synthetic visit: fatigue       65+  Female   
2  2024-05-01  SYN000213   08104  Synthetic visit: headache     18–44    Male   
3  2024-05-01  SYN000379   08104   Synthetic visit: fatigue      0–17  Female   
4  2024-05-01  SYN000071   08102   Synthetic visit: fatigue       65+  Female   

      race  
0    Black  
1    Asian  
2  Unknown  
3    Other  
4    Asian  


In [6]:
health_data = pd.read_csv(
    OUTPUT_FILE,
    dtype={"zipcode": str, "patient_id": str},
    parse_dates=["date"]
)

In [10]:
# Work on a copy and standardize dates
visits = health_data.copy()
visits["date"] = pd.to_datetime(visits["date"]).dt.normalize()

# Turn demographic categories into separate indicator columns.
# Each visit contributes 1 to its corresponding categories.
age_columns = pd.get_dummies(
    visits["age_group"],
    dtype=int
).reindex(
    columns=["0–17", "18–44", "45–64", "65+"],
    fill_value=0
)

age_columns.columns = [
    "age_0_17",
    "age_18_44",
    "age_45_64",
    "age_65_plus"
]

sex_columns = pd.get_dummies(
    visits["sex"],
    dtype=int
).reindex(
    columns=["Female", "Male", "Unknown"],
    fill_value=0
)

sex_columns.columns = [
    "sex_Female",
    "sex_Male",
    "sex_Unknown"
]

race_columns = pd.get_dummies(
    visits["race"],
    dtype=int
).reindex(
    columns=["White", "Black", "Asian", "Other", "Unknown"],
    fill_value=0
)

race_columns.columns = [
    "race_White",
    "race_Black",
    "race_Asian",
    "race_Other",
    "race_Unknown"
]

# Combine indicators with one total-visit indicator per record
daily_records = pd.concat(
    [
        visits[["date"]].reset_index(drop=True),
        age_columns.reset_index(drop=True),
        sex_columns.reset_index(drop=True),
        race_columns.reset_index(drop=True)
    ],
    axis=1
)

daily_records["visits"] = 1

# Sum visits and demographic indicators by date
daily_visits = (
    daily_records
    .groupby("date")
    .sum()
    .reindex(
        pd.date_range(START_DATE, END_DATE, freq="D"),
        fill_value=0
    )
    .rename_axis("date")
    .reset_index()
)

# Put total visits immediately after the date
daily_visits = daily_visits[
    ["date", "visits"]
    + list(age_columns.columns)
    + list(sex_columns.columns)
    + list(race_columns.columns)
]

# Save
daily_visits.to_csv(
    OUTPUT_FILE.parent / "synthetic_daily_demographics.csv",
    index=False
)

print(daily_visits.head())

        date  visits  age_0_17  age_18_44  age_45_64  age_65_plus  sex_Female  \
0 2024-05-01      13         3          4          4            2           7   
1 2024-05-02       5         1          1          2            1           2   
2 2024-05-03       7         2          2          1            2           3   
3 2024-05-04      14         4          5          1            4           6   
4 2024-05-05       9         3          4          1            1           3   

   sex_Male  sex_Unknown  race_White  race_Black  race_Asian  race_Other  \
0         5            1           3           2           3           2   
1         2            1           0           1           1           2   
2         2            2           2           3           0           2   
3         2            6           2           4           1           2   
4         1            5           1           2           4           2   

   race_Unknown  
0             3  
1             1  
2 

In [11]:
daily_visits = daily_visits.copy()
daily_visits["date"] = pd.to_datetime(daily_visits["date"])

day_of_year = daily_visits["date"].dt.dayofyear.to_numpy()

is_weekend = (
    daily_visits["date"].dt.dayofweek >= 5
).to_numpy().astype(float)

daily_visits["baseline_visits"] = (
    100
    + 8 * np.cos(
        2 * np.pi * (day_of_year - 15) / 365.25
    )
    - 6 * is_weekend
).round(3)

# Save ALL columns: visits, demographic counts, and baseline
daily_visits.to_csv(
    OUTPUT_FILE.parent / "daily_health_sample.csv",
    index=False
)

print(daily_visits.head())

        date  visits  age_0_17  age_18_44  age_45_64  age_65_plus  sex_Female  \
0 2024-05-01      13         3          4          4            2           7   
1 2024-05-02       5         1          1          2            1           2   
2 2024-05-03       7         2          2          1            2           3   
3 2024-05-04      14         4          5          1            4           6   
4 2024-05-05       9         3          4          1            1           3   

   sex_Male  sex_Unknown  race_White  race_Black  race_Asian  race_Other  \
0         5            1           3           2           3           2   
1         2            1           0           1           1           2   
2         2            2           2           3           0           2   
3         2            6           2           4           1           2   
4         1            5           1           2           4           2   

   race_Unknown  baseline_visits  
0             3      